In [1]:
from pathlib import Path
import pandas as pd
import os
import re
from itertools import combinations
import json

In [2]:
data_dir = '../data/orig/shots_dataset/'
output_dir = '../data/merged_v2/'

In [3]:
# Optional download raw dataset

#import kagglehub
#path = kagglehub.dataset_download(
#    "brains14482/nba-playbyplay-and-shotdetails-data-19962021",
#    output_dir= data_dir
#)

In [4]:
def parse_filename(fname, prefix='shotdetail_'):
    # Helper to extract year and playoffs
    m = re.match(prefix + r"(po_)?(\d{4})\.csv", fname)
    if m:
        is_po = m.group(1) is not None
        year = int(m.group(2))
        return year, is_po
    return None

In [5]:
#main_table = 'shotdetail_'
#tables2merge = ['nbastatsv3_', 'nbastats_'] # tables to merge with 'shotdetail_'
main_table = 'nbastatsv3_'
tables2merge = ['shotdetail_', 'nbastats_'] # tables to merge with 'nbastatsv3_'

merge_keys = {
    'shotdetail_': ['GAME_ID', 'GAME_EVENT_ID'], 
    'nbastatsv3_': ['gameId', 'actionNumber'],
    'nbastats_': ['GAME_ID', 'EVENTNUM']
}

In [6]:
shot_files = [
    f for f in os.listdir(data_dir)
    if f.startswith(main_table) and f.endswith(".csv")
]

shot_files = sorted(shot_files, key=lambda f: parse_filename(f, main_table))


In [10]:
df_merged = None
count_rows_addded = {}
duplicated_columns = {}

for shot_file in shot_files:
    year, is_po = parse_filename(shot_file, main_table)
    suffix = f"po_{year}" if is_po else f"{year}"

    df_year_merged = pd.read_csv(os.path.join(data_dir, shot_file))
    base_len = len(df_year_merged.index)

    # horizontal merges
    for tabletype in tables2merge:
        other_file = f"{tabletype}{suffix}.csv"
        path = os.path.join(data_dir, other_file)

        if not os.path.exists(path):
            continue

        df_other = pd.read_csv(path)

        df_year_merged = pd.merge(
            df_year_merged,
            df_other,
            how='left',
            left_on=merge_keys[main_table],
            right_on=merge_keys[tabletype]
        )

    # metadata
    df_year_merged["year"] = year
    df_year_merged["is_playoffs"] = is_po
    
    count_rows_addded[suffix] = len(df_year_merged.index) - base_len
    duplicated_columns[suffix] = [(i, j) for i,j in combinations(df_year_merged, 2) if df_year_merged[i].equals(df_year_merged[j])]

    # vertical merge
    if df_merged is None:
        df_merged = df_year_merged
    else:
        df_merged = pd.concat([df_merged, df_year_merged], ignore_index=True)



In [11]:
df_merged.info()

<class 'pandas.DataFrame'>
RangeIndex: 17613753 entries, 0 to 17613752
Data columns (total 84 columns):
 #   Column                     Dtype  
---  ------                     -----  
 0   actionNumber               int64  
 1   clock                      str    
 2   period                     int64  
 3   teamId                     int64  
 4   teamTricode                str    
 5   personId                   int64  
 6   playerName                 str    
 7   playerNameI                str    
 8   xLegacy                    int64  
 9   yLegacy                    int64  
 10  shotDistance               int64  
 11  shotResult                 str    
 12  isFieldGoal                int64  
 13  scoreHome                  float64
 14  scoreAway                  float64
 15  pointsTotal                int64  
 16  location                   str    
 17  description                str    
 18  actionType                 str    
 19  subType                    str    
 20  videoAvaila

# Save Metadata and combined table

In [12]:
#df_merged.to_csv(Path(output_dir) / "merged_dataset.csv", index=False)

meta_data = {
    "count_rows_added": count_rows_addded,
    "duplicated_columns": duplicated_columns
}

with open(Path(output_dir) / 'meta_data.json', "w") as f:
    json.dump(meta_data, f)

In [14]:
#df_merged = pd.read_csv(Path(output_dir) / "merged_dataset.csv")
df_merged.to_parquet(Path(output_dir) / "merged_dataset.parquet", index=False)

In [15]:
meta_data

{'count_rows_added': {'1996': 0,
  'po_1996': 0,
  '1997': 0,
  'po_1997': 0,
  '1998': 0,
  'po_1998': 0,
  '1999': 0,
  'po_1999': 0,
  '2000': 0,
  'po_2000': 0,
  '2001': 0,
  'po_2001': 0,
  '2002': 0,
  'po_2002': 0,
  '2003': 0,
  'po_2003': 0,
  '2004': 0,
  'po_2004': 0,
  '2005': 0,
  'po_2005': 0,
  '2006': 0,
  'po_2006': 0,
  '2007': 0,
  'po_2007': 0,
  '2008': 0,
  'po_2008': 0,
  '2009': 0,
  'po_2009': 0,
  '2010': 0,
  'po_2010': 0,
  '2011': 0,
  'po_2011': 0,
  '2012': 0,
  'po_2012': 0,
  '2013': 0,
  'po_2013': 0,
  '2014': 162,
  'po_2014': 12,
  '2015': 109,
  'po_2015': 3,
  '2016': 112,
  'po_2016': 8,
  '2017': 155,
  'po_2017': 15,
  '2018': 143,
  'po_2018': 12,
  '2019': 70,
  'po_2019': 5,
  '2020': 95,
  'po_2020': 1,
  '2021': 11,
  'po_2021': 0,
  '2022': 19,
  'po_2022': 0,
  '2023': 3,
  'po_2023': 0,
  '2024': 2,
  'po_2024': 0},
 'duplicated_columns': {'1996': [],
  'po_1996': [('actionNumber', 'EVENTNUM'),
   ('period', 'PERIOD_y'),
   ('videoAvai

# Initial preprocessing to reduce amount of data

In [4]:
df_merged = pd.read_parquet(Path(output_dir) / "merged_dataset.parquet")

In [16]:
df_merged.actionType.unique()

<ArrowStringArray>
[                                  'period',
                                'Jump Ball',
                                'Made Shot',
                                     'Foul',
                               'Free Throw',
                              'Missed Shot',
                                  'Rebound',
                                 'Turnover',
                                        nan,
                                  'Timeout',
                             'Substitution',
                                'Violation',
                                 'Ejection',
 'Timeout                                 ',
 'Turnover                                ',
 'Ejection                                ',
 'Made Shot                               ',
 'Missed Shot                             ',
 'Violation                               ',
                           'Instant Replay',
 'Foul                                    ',
 'Instant Replay                    

# Handle free throws

In [5]:
df_freethrow = df_merged.loc[df_merged['actionType'] == 'Free Throw']

In [6]:
pd.options.display.max_columns = None
df_freethrow.head(10)

,actionNumber,clock,period,teamId,teamTricode,personId,playerName,playerNameI,xLegacy,yLegacy,shotDistance,shotResult,isFieldGoal,scoreHome,scoreAway,pointsTotal,location,description,actionType,subType,videoAvailable,actionId,gameId,GRID_TYPE,GAME_ID_x,GAME_EVENT_ID,PLAYER_ID,PLAYER_NAME,TEAM_ID,TEAM_NAME,PERIOD_x,MINUTES_REMAINING,SECONDS_REMAINING,EVENT_TYPE,ACTION_TYPE,SHOT_TYPE,SHOT_ZONE_BASIC,SHOT_ZONE_AREA,SHOT_ZONE_RANGE,SHOT_DISTANCE,LOC_X,LOC_Y,SHOT_ATTEMPTED_FLAG,SHOT_MADE_FLAG,GAME_DATE,HTM,VTM,GAME_ID_y,EVENTNUM,EVENTMSGTYPE,EVENTMSGACTIONTYPE,PERIOD_y,WCTIMESTRING,PCTIMESTRING,HOMEDESCRIPTION,NEUTRALDESCRIPTION,VISITORDESCRIPTION,SCORE,SCOREMARGIN,PERSON1TYPE,PLAYER1_ID,PLAYER1_NAME,PLAYER1_TEAM_ID,PLAYER1_TEAM_CITY,PLAYER1_TEAM_NICKNAME,PLAYER1_TEAM_ABBREVIATION,PERSON2TYPE,PLAYER2_ID,PLAYER2_NAME,PLAYER2_TEAM_ID,PLAYER2_TEAM_CITY,PLAYER2_TEAM_NICKNAME,PLAYER2_TEAM_ABBREVIATION,PERSON3TYPE,PLAYER3_ID,PLAYER3_NAME,PLAYER3_TEAM_ID,PLAYER3_TEAM_CITY,PLAYER3_TEAM_NICKNAME,PLAYER3_TEAM_ABBREVIATION,VIDEO_AVAILABLE_FLAG,year,is_playoffs,shotValue
4,6,PT11M39.00S,1,1610612741,CHI,23,Rodman,D. Rodman,0,0,0,NaN,0,0.0,3.0,3,v,Rodman Free Throw 1 of 1 (3 PTS),Free Throw,Free Throw 1 of 1,0,5,29600001,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,29600001.0,6.0,3.0,10.0,1.0,11:17 PM,11:39,NaN,NaN,Rodman Free Throw 1 of 1 (3 PTS),3 - 0,-3,5.0,23.0,Dennis Rodman,1.610613e+09,Chicago,Bulls,CHI,0.0,0.0,NaN,NaN,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,0.0,1996,False,NaN
22,21,PT09M06.00S,1,1610612738,BOS,442,Ellison,P. Ellison,0,0,0,NaN,0,7.0,9.0,16,h,Ellison Free Throw 1 of 1 (3 PTS),Free Throw,Free Throw 1 of 1,0,23,29600001,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,29600001.0,21.0,3.0,10.0,1.0,11:20 PM,9:06,Ellison Free Throw 1 of 1 (3 PTS),NaN,NaN,9 - 7,-2,4.0,442.0,Pervis Ellison,1.610613e+09,Boston,Celtics,BOS,0.0,0.0,NaN,NaN,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,0.0,1996,False,NaN
36,35,PT07M20.00S,1,1610612738,BOS,296,Fox,R. Fox,0,0,0,NaN,0,0.0,0.0,0,h,MISS Fox Free Throw Technical,Free Throw,Free Throw Technical,0,37,29600001,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,29600001.0,35.0,3.0,16.0,1.0,11:23 PM,7:20,MISS Fox Free Throw Technical,NaN,NaN,NaN,NaN,4.0,296.0,Rick Fox,1.610613e+09,Boston,Celtics,BOS,0.0,0.0,NaN,NaN,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,0.0,1996,False,NaN
75,72,PT02M59.00S,1,1610612741,CHI,893,Jordan,M. Jordan,0,0,0,NaN,0,21.0,18.0,39,v,Jordan Free Throw 1 of 2 (5 PTS),Free Throw,Free Throw 1 of 2,0,76,29600001,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,29600001.0,72.0,3.0,11.0,1.0,11:35 PM,2:59,NaN,NaN,Jordan Free Throw 1 of 2 (5 PTS),18 - 21,3,5.0,893.0,Michael Jordan,1.610613e+09,Chicago,Bulls,CHI,0.0,0.0,NaN,NaN,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,0.0,1996,False,NaN
78,75,PT02M59.00S,1,1610612741,CHI,893,Jordan,M. Jordan,0,0,0,NaN,0,21.0,19.0,40,v,Jordan Free Throw 2 of 2 (6 PTS),Free Throw,Free Throw 2 of 2,0,79,29600001,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,29600001.0,75.0,3.0,12.0,1.0,11:35 PM,2:59,NaN,NaN,Jordan Free Throw 2 of 2 (6 PTS),19 - 21,2,5.0,893.0,Michael Jordan,1.610613e+09,Chicago,Bulls,CHI,0.0,0.0,NaN,NaN,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,0.0,1996,False,NaN
90,86,PT01M38.00S,1,1610612738,BOS,783,Brickowski,F. Brickowski,0,0,0,NaN,0,0.0,0.0,0,h,MISS Brickowski Free Throw 1 of 2,Free Throw,Free Throw 1 of 2,0,91,29600001,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,29600001.0,86.0,3.0,11.0,1.0,11:37 PM,1:38,MISS Brickowski Free Throw 1 of 2,NaN,NaN,NaN,NaN,4.0,783.0,Frank Brickowski,1.610613e+09,Boston,Celtics,BOS,0.0,0.0,NaN,NaN,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,0.0,1996,False,NaN
92,88,PT01M38.00S,1,1610612738,BOS,783,Brickowski,F. Brickowski,0,0,

We can use the description column to identify the result of a free throw. They all seem to contain "MISS"

In [12]:
missed_freethrows = df_freethrow['description'].apply(lambda x: 'MISS' in x)
sum(missed_freethrows)/(len(missed_freethrows))

0.24113145205975595

The average miss rate seems to be realistic. We can use this condition to identify the result of the free throws

## Add information to the columns with nan, if possible

In [20]:
# Get index of free throws and misses
idx_freethrows = (df_merged['actionType'] == 'Free Throw')
idx_freethrows_missed = ((df_merged['actionType'] == 'Free Throw') & (df_merged['description'].str.contains('MISS', na=False)))
idx_freethrows_hit = (idx_freethrows) & ~(idx_freethrows_missed)

In [34]:
# Position related fields
df_merged.loc[idx_freethrows, ['LOC_X']] = 0
df_merged.loc[idx_freethrows, ['yLegacy', 'LOC_Y', 'shotDistance', 'SHOT_DISTANCE']] = 15 # free throws at 15 ft from the basket

# Meta info (id ...)
df_merged.loc[idx_freethrows, ['GAME_ID_x']] = df_merged.loc[idx_freethrows, 'gameId']
df_merged.loc[idx_freethrows, ['GAME_EVENT_ID']] = df_merged.loc[idx_freethrows, 'actionNumber']
df_merged.loc[idx_freethrows, ['PLAYER_ID']] = df_merged.loc[idx_freethrows, 'personId']
df_merged.loc[idx_freethrows, ['PLAYER_NAME']] = df_merged.loc[idx_freethrows, 'PLAYER1_NAME']
df_merged.loc[idx_freethrows, ['TEAM_ID']] = df_merged.loc[idx_freethrows, 'teamId']
df_merged.loc[idx_freethrows, ['TEAM_NAME']] = df_merged.loc[idx_freethrows, 'PLAYER1_TEAM_CITY'] + ' ' + df_merged.loc[idx_freethrows, 'PLAYER1_TEAM_NICKNAME']
df_merged.loc[idx_freethrows, ['PERIOD_x']] = df_merged.loc[idx_freethrows, 'period']

#df_merged.loc[idx_freethrows, ['MINUTES_REMAINING', 'SECONDS_REMAINING']] = df_merged['PCTIMESTRING'].str.split(':', expand=True).astype(int)
split_cols = df_merged.loc[idx_freethrows, 'PCTIMESTRING'].str.split(':', expand=True)
df_merged.loc[idx_freethrows, ['MINUTES_REMAINING', 'SECONDS_REMAINING']] = (split_cols.apply(pd.to_numeric, downcast='integer'))

df_merged.loc[idx_freethrows, ['ACTION_TYPE']] = 'Free Throw'
df_merged.loc[idx_freethrows, ['SHOT_TYPE']] = '1PT Free Throw'

df_merged.loc[idx_freethrows, ['SHOT_ZONE_BASIC']] = 'Mid-Range'
df_merged.loc[idx_freethrows, ['SHOT_ZONE_AREA']] = 'Center(C)'
df_merged.loc[idx_freethrows, ['SHOT_ZONE_RANGE']] =  '8-16 ft.'
df_merged.loc[idx_freethrows, ['SHOT_ATTEMPTED_FLAG']] = 1

#df_merged.loc[idx_freethrows, ['GAME_DATE']] = --> No Game Date availble!

# HTM and VTM cannot not easily be derived 
#df_merged.loc[idx_freethrows, ['HTM']] = 
#df_merged.loc[idx_freethrows, ['VTM']] = 

# Result dependent columns
# for hits
df_merged.loc[idx_freethrows_hit, ['shotResult']] = 'Made'
df_merged.loc[idx_freethrows_hit, ['EVENT_TYPE']] = 'Made Shot'
df_merged.loc[idx_freethrows_hit, ['SHOT_MADE_FLAG']] = 1

# for misses
df_merged.loc[idx_freethrows_missed, ['shotResult']] = 'Missed'
df_merged.loc[idx_freethrows_missed, ['EVENT_TYPE']] = 'Missed Shot'
df_merged.loc[idx_freethrows_missed, ['SHOT_MADE_FLAG']] = 0
 	


In [35]:
df_merged.loc[idx_freethrows].head(2)

,actionNumber,clock,period,teamId,teamTricode,personId,playerName,playerNameI,xLegacy,yLegacy,shotDistance,shotResult,isFieldGoal,scoreHome,scoreAway,pointsTotal,location,description,actionType,subType,videoAvailable,actionId,gameId,GRID_TYPE,GAME_ID_x,GAME_EVENT_ID,PLAYER_ID,PLAYER_NAME,TEAM_ID,TEAM_NAME,PERIOD_x,MINUTES_REMAINING,SECONDS_REMAINING,EVENT_TYPE,ACTION_TYPE,SHOT_TYPE,SHOT_ZONE_BASIC,SHOT_ZONE_AREA,SHOT_ZONE_RANGE,SHOT_DISTANCE,LOC_X,LOC_Y,SHOT_ATTEMPTED_FLAG,SHOT_MADE_FLAG,GAME_DATE,HTM,VTM,GAME_ID_y,EVENTNUM,EVENTMSGTYPE,EVENTMSGACTIONTYPE,PERIOD_y,WCTIMESTRING,PCTIMESTRING,HOMEDESCRIPTION,NEUTRALDESCRIPTION,VISITORDESCRIPTION,SCORE,SCOREMARGIN,PERSON1TYPE,PLAYER1_ID,PLAYER1_NAME,PLAYER1_TEAM_ID,PLAYER1_TEAM_CITY,PLAYER1_TEAM_NICKNAME,PLAYER1_TEAM_ABBREVIATION,PERSON2TYPE,PLAYER2_ID,PLAYER2_NAME,PLAYER2_TEAM_ID,PLAYER2_TEAM_CITY,PLAYER2_TEAM_NICKNAME,PLAYER2_TEAM_ABBREVIATION,PERSON3TYPE,PLAYER3_ID,PLAYER3_NAME,PLAYER3_TEAM_ID,PLAYER3_TEAM_CITY,PLAYER3_TEAM_NICKNAME,PLAYER3_TEAM_ABBREVIATION,VIDEO_AVAILABLE_FLAG,year,is_playoffs,shotValue
4,6,PT11M39.00S,1,1610612741,CHI,23,Rodman,D. Rodman,0,15,15,Made,0,0.0,3.0,3,v,Rodman Free Throw 1 of 1 (3 PTS),Free Throw,Free Throw 1 of 1,0,5,29600001,NaN,29600001.0,6.0,23.0,Dennis Rodman,1.610613e+09,Chicago Bulls,1.0,NaN,NaN,Made Shot,Free Throw,1PT Free Throw,Mid-Range,Center(C),8-16 ft.,15.0,0.0,15.0,1.0,1.0,NaN,NaN,NaN,29600001.0,6.0,3.0,10.0,1.0,11:17 PM,11:39,NaN,NaN,Rodman Free Throw 1 of 1 (3 PTS),3 - 0,-3,5.0,23.0,Dennis Rodman,1.610613e+09,Chicago,Bulls,CHI,0.0,0.0,NaN,NaN,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,0.0,1996,False,NaN
22,21,PT09M06.00S,1,1610612738,BOS,442,Ellison,P. Ellison,0,15,15,Made,0,7.0,9.0,16,h,Ellison Free Throw 1 of 1 (3 PTS),Free Throw,Free Throw 1 of 1,0,23,29600001,NaN,29600001.0,21.0,442.0,Pervis Ellison,1.610613e+09,Boston Celtics,1.0,NaN,NaN,Made Shot,Free Throw,1PT Free Throw,Mid-Range,Center(C),8-16 ft.,15.0,0.0,15.0,1.0,1.0,NaN,NaN,NaN,29600001.0,21.0,3.0,10.0,1.0,11:20 PM,9:06,Ellison Free Throw 1 of 1 (3 PTS),NaN,NaN,9 - 7,-2,4.0,442.0,Pervis Ellison,1.610613e+09,Boston,Celtics,BOS,0.0,0.0,NaN,NaN,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,0.0,1996,False,NaN


# Check possible target columns and drop rows without result, for most promising

In [41]:
df_merged[['SHOT_MADE_FLAG', 'shotResult', 'EVENT_TYPE']].isna().sum()

SHOT_MADE_FLAG    9405793
shotResult        9764837
EVENT_TYPE        9405793
dtype: int64

In [42]:
idx_no_target = df_merged[['SHOT_MADE_FLAG', 'shotResult', 'EVENT_TYPE']].isna().all(axis=1)

In [46]:
df_merged.loc[idx_no_target, 'actionType'].value_counts()

actionType
Rebound                                     3744043
Foul                                        1587530
Substitution                                1521203
Turnover                                    1058210
Timeout                                      465422
period                                       298175
Violation                                     64769
Jump Ball                                     64040
Instant Replay                                37211
Ejection                                       2434
Foul                                            561
Turnover                                        475
Instant Replay                                  394
Timeout                                         102
Ejection                                         20
Violation                                         5
Missed Shot                                       1
Name: count, dtype: int64

The most rows without target are no real attempts (timeouts, fouls, ...). It should be fine to drop them.
There is only one "missed shot" without possible target, it should be fine to drop it

## Drop rows without any possible target variable

In [48]:
print(df_merged.shape)
df_merged = df_merged[~idx_no_target]
print(df_merged.shape)

(17613753, 84)
(8208684, 84)


In [50]:
df_merged['actionType'].value_counts(dropna=False)

actionType
Missed Shot                                 3320517
Made Shot                                   2772111
Free Throw                                  1756009
NaN                                          359710
Made Shot                                       153
Missed Shot                                     126
Rebound                                          24
Foul                                             12
Turnover                                         10
Substitution                                      7
Violation                                         2
Timeout                                           2
period                                            1
Name: count, dtype: int64

We can also drop rows with actions, not related to shots (foul, timeout, ...)

In [55]:
actions_to_drop = ['Rebound', 'Foul', 'Turnover', 'Substitution', 'Violation', 'Timeout', 'period']
df_merged = df_merged[~ (df_merged['actionType'].isin(actions_to_drop))]

In [57]:
df_merged.loc[df_merged['actionType'].isna(), 'ACTION_TYPE'].unique()

<ArrowStringArray>
[                        'Layup Shot',                 'Driving Layup Shot',
                          'Jump Shot',                          'Hook Shot',
                  'Running Jump Shot',                          'Dunk Shot',
                     'Slam Dunk Shot',                           'Tip Shot',
                  'Driving Dunk Shot',                            'No Shot',
                 'Running Layup Shot',               'Turnaround Jump Shot',
                 'Reverse Layup Shot',                   'Running Tip Shot',
                  'Reverse Dunk Shot',           'Driving Finger Roll Shot',
                   'Finger Roll Shot',               'Turnaround Hook Shot',
                  'Running Hook Shot',                  'Driving Hook Shot',
           'Running Finger Roll Shot',               'Alley Oop Layup shot',
                  'Running Dunk Shot',                 'Fadeaway Jump Shot',
                     'Jump Hook Shot',                'Al

The initial filtering should be fine for now to have the major amount of rows excluded that are no shots.
Further filtering can be done in the preprocessing step

In [58]:
# Save the dataframe
df_merged.to_parquet(Path(output_dir) / "merged_dataset.parquet", index=False)